# 04 Statistical Analysis: Odds Ratios

This notebook estimates associations between candidate clinical predictors and the binary `Outcome` using logistic regression models fit with `statsmodels`.

The workflow includes:

1. Unadjusted logistic regression models for individual predictors.
2. An adjusted logistic regression model with clinically relevant predictors.
3. Odds ratios, 95% confidence intervals, and p-values.
4. A clean results table saved to `reports/odds_ratio_results.csv`.

> **Interpretation note:** These models estimate statistical associations in an observational dataset. Odds ratios from these models should **not** be interpreted as causal effects. Causal interpretation would require stronger assumptions, careful study design, temporal validation, and control for confounding and selection mechanisms beyond the scope of this notebook.


## Setup

Load the processed cardiac patient dataset and configure the paths used by this analysis.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cardiac_patient_processed.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
OUTPUT_PATH = REPORTS_DIR / "odds_ratio_results.csv"

OUTCOME_COL = "Outcome"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:0.4f}")


## Load data

The analysis uses the processed dataset so engineered clinical features are available if needed. `Outcome` must be binary-coded as 0/1 for logistic regression.


In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print("Outcome distribution:")
display(df[OUTCOME_COL].value_counts(dropna=False).rename_axis("Outcome").to_frame("count"))
display(df.head())


Dataset shape: 5,906 rows x 30 columns
Outcome distribution:


,count
Outcome,
1,5053
0,853


,ID,SBP,DBP,HR,RR,BT,SpO2,Age,Gender,GCS,Na,K,Cl,Urea,Ceratinine,Alcoholic,Smoke,FHCD,TriageScore,Outcome,pulse_pressure,shock_index,age_band,hypoxemia_flag,gcs_severity,sodium_abnormal_flag,potassium_abnormal_flag,chloride_abnormal_flag,urea_abnormal_flag,creatinine_abnormal_flag
0,1,163,95,90,18,98,98,66,1,15,139.0000,4.0000,105.0000,41.0000,91.0000,1.0000,1.0000,0.0000,3.0000,1,68,0.5521,65-79,0,mild,0.0000,0.0000,0.0000,0.0000,0.0000
1,1,134,85,85,15,98,98,66,1,15,139.0000,4.0000,105.0000,41.0000,91.0000,1.0000,1.0000,0.0000,3.0000,1,49,0.6343,65-79,0,mild,0.0000,0.0000,0.0000,0.0000,0.0000
2,1,121,77,80,19,98,98,66,1,15,139.0000,4.0000,105.0000,41.0000,91.0000,1.0000,1.0000,0.0000,3.0000,1,44,0.6612,65-79,0,mild,0.0000,0.0000,0.0000,0.0000,0.0000
3,1,103,78,70,16,98,98,66,1,15,139.0000,4.0000,105.0000,41.0000,91.0000,1.0000,1.0000,0.0000,3.0000,1,25,0.6796,65-79,0,mild,0.0000,0.0000,0.0000,0.0000,0.0000
4,1,96,70,59,13,98,98,66,1,15,139.0000,4.0000,105.0000,41.0000,91.0000,1.0000,1.0000,0.0000,3.0000,1,26,0.6146,65-79,0,mild,0.0000,0.0000,0.0000,0.0000,0.0000


## Predictor specification

The unadjusted models estimate one predictor at a time. The adjusted model includes clinically relevant demographics, vital signs, neurologic status, triage acuity, and history variables selected a priori for interpretability rather than automated feature selection.

Categorical predictors are modeled with `C(...)` in the `statsmodels` formula interface. The first sorted category is the reference level, so categorical odds ratios compare each displayed level with that reference.


In [3]:
continuous_predictors = [
    "Age",
    "SBP",
    "HR",
    "RR",
    "SpO2",
    "GCS",
]

categorical_predictors = [
    "Gender",
    "TriageScore",
    "Alcoholic",
    "Smoke",
    "FHCD",
]

predictor_metadata = {
    "Age": {"type": "continuous", "label": "Age", "comparison": "per 1-year increase"},
    "SBP": {"type": "continuous", "label": "Systolic blood pressure", "comparison": "per 1 mmHg increase"},
    "HR": {"type": "continuous", "label": "Heart rate", "comparison": "per 1 bpm increase"},
    "RR": {"type": "continuous", "label": "Respiratory rate", "comparison": "per 1 breath/min increase"},
    "SpO2": {"type": "continuous", "label": "Oxygen saturation", "comparison": "per 1 percentage-point increase"},
    "GCS": {"type": "continuous", "label": "Glasgow Coma Scale", "comparison": "per 1-point increase"},
    "Gender": {"type": "categorical", "label": "Gender", "comparison": "level vs reference"},
    "TriageScore": {"type": "categorical", "label": "Triage score", "comparison": "level vs reference"},
    "Alcoholic": {"type": "categorical", "label": "Alcohol history", "comparison": "level vs reference"},
    "Smoke": {"type": "categorical", "label": "Smoking history", "comparison": "level vs reference"},
    "FHCD": {"type": "categorical", "label": "Family history of cardiac disease", "comparison": "level vs reference"},
}

unadjusted_predictors = continuous_predictors + categorical_predictors
adjusted_predictors = unadjusted_predictors.copy()

print("Unadjusted predictors:", unadjusted_predictors)
print("Adjusted predictors:", adjusted_predictors)


Unadjusted predictors: ['Age', 'SBP', 'HR', 'RR', 'SpO2', 'GCS', 'Gender', 'TriageScore', 'Alcoholic', 'Smoke', 'FHCD']
Adjusted predictors: ['Age', 'SBP', 'HR', 'RR', 'SpO2', 'GCS', 'Gender', 'TriageScore', 'Alcoholic', 'Smoke', 'FHCD']


## Helper functions

The functions below fit logistic regression models, convert coefficients to odds ratios, compute Wald 95% confidence intervals on the log-odds scale, and format the model terms for a clean results table.


In [4]:
def formula_term(predictor: str) -> str:
    """Return a statsmodels formula term for a configured predictor."""
    if predictor_metadata[predictor]["type"] == "categorical":
        return f"C({predictor})"
    return predictor


def build_formula(predictors: list[str]) -> str:
    """Build a logistic regression formula for the selected predictors."""
    return f"{OUTCOME_COL} ~ " + " + ".join(formula_term(predictor) for predictor in predictors)


def predictor_from_term(term: str) -> str:
    """Map a statsmodels coefficient term back to the source predictor name."""
    if term.startswith("C("):
        return term.split(")", 1)[0].replace("C(", "")
    return term


def comparison_from_term(term: str) -> str:
    """Create a human-readable odds-ratio comparison for a coefficient term."""
    if term.startswith("C(") and "[T." in term:
        predictor = predictor_from_term(term)
        level = term.split("[T.", 1)[1].rstrip("]")
        return f"level {level} vs reference"
    predictor = predictor_from_term(term)
    return predictor_metadata[predictor]["comparison"]


def fit_logistic_model(model_name: str, predictors: list[str]) -> tuple[object, pd.DataFrame]:
    """Fit a statsmodels logistic regression model and return tidy OR results."""
    model_columns = [OUTCOME_COL] + predictors
    model_data = df[model_columns].dropna().copy()
    model_data[OUTCOME_COL] = model_data[OUTCOME_COL].astype(int)

    formula = build_formula(predictors)
    result = smf.logit(formula=formula, data=model_data).fit(disp=False, maxiter=200)

    coefficient_table = pd.DataFrame({
        "term": result.params.index,
        "coefficient": result.params.values,
        "std_error": result.bse.values,
        "p_value": result.pvalues.values,
    })
    conf_int = result.conf_int()
    coefficient_table["ci_lower_logit"] = conf_int[0].values
    coefficient_table["ci_upper_logit"] = conf_int[1].values

    coefficient_table = coefficient_table[coefficient_table["term"] != "Intercept"].copy()
    coefficient_table["model"] = model_name
    coefficient_table["predictor"] = coefficient_table["term"].map(predictor_from_term)
    coefficient_table["predictor_label"] = coefficient_table["predictor"].map(lambda predictor: predictor_metadata[predictor]["label"])
    coefficient_table["comparison"] = coefficient_table["term"].map(comparison_from_term)
    coefficient_table["n_obs"] = int(result.nobs)
    coefficient_table["events"] = int(model_data[OUTCOME_COL].sum())
    coefficient_table["non_events"] = int((model_data[OUTCOME_COL] == 0).sum())
    coefficient_table["odds_ratio"] = np.exp(coefficient_table["coefficient"])
    coefficient_table["ci_lower"] = np.exp(coefficient_table["ci_lower_logit"])
    coefficient_table["ci_upper"] = np.exp(coefficient_table["ci_upper_logit"])

    ordered_columns = [
        "model",
        "predictor",
        "predictor_label",
        "term",
        "comparison",
        "n_obs",
        "events",
        "non_events",
        "coefficient",
        "std_error",
        "odds_ratio",
        "ci_lower",
        "ci_upper",
        "p_value",
    ]
    return result, coefficient_table[ordered_columns]


## Unadjusted logistic regression models

Each unadjusted model includes `Outcome` and one predictor. These models describe marginal associations and do not account for confounding by other patient characteristics.


In [5]:
unadjusted_results = []
unadjusted_model_objects = {}

for predictor in unadjusted_predictors:
    model_name = f"Unadjusted: {predictor}"
    result, tidy_table = fit_logistic_model(model_name=model_name, predictors=[predictor])
    unadjusted_model_objects[predictor] = result
    unadjusted_results.append(tidy_table)

unadjusted_results = pd.concat(unadjusted_results, ignore_index=True)

unadjusted_display = unadjusted_results.copy()
for column in ["coefficient", "std_error", "odds_ratio", "ci_lower", "ci_upper", "p_value"]:
    unadjusted_display[column] = unadjusted_display[column].map(lambda value: f"{value:0.4g}")

display(unadjusted_display)


,model,predictor,predictor_label,term,comparison,n_obs,events,non_events,coefficient,std_error,odds_ratio,ci_lower,ci_upper,p_value
0,Unadjusted: Age,Age,Age,Age,per 1-year increase,5906,5053,853,-0.05179,0.003136,0.9495,0.9437,0.9554,2.948e-61
1,Unadjusted: SBP,SBP,Systolic blood pressure,SBP,per 1 mmHg increase,5906,5053,853,0.03239,0.00198,1.033,1.029,1.037,3.849e-60
2,Unadjusted: HR,HR,Heart rate,HR,per 1 bpm increase,5906,5053,853,-0.07614,0.002773,0.9267,0.9217,0.9317,5.364e-166
3,Unadjusted: RR,RR,Respiratory rate,RR,per 1 breath/min increase,5906,5053,853,-0.134,0.007058,0.8746,0.8625,0.8867,2.111e-80
4,Unadjusted: SpO2,SpO2,Oxygen saturation,SpO2,per 1 percentage-point increase,5906,5053,853,0.4088,0.01965,1.505,1.448,1.564,3.754e-96
5,Unadjusted: GCS,GCS,Glasgow Coma Scale,GCS,per 1-point increase,5906,5053,853,-0.02468,0.003555,0.9756,0.9689,0.9824,3.875e-12
6,Unadjusted: Gender,Gender,Gender,C(Gender)[T.1],level 1 vs reference,5906,5053,853,-0.0133,0.08148,0.9868,0.8411,1.158,0.8703
7,Unadjusted: TriageScore,TriageScore,Triage score,C(TriageScore)[T.2.0],level 2.0 vs reference,4003,3633,370,-3.585,1.004,0.02775,0.003875,0.1986,0.000358
8,Unadjusted: TriageScore,TriageScore,Triage score,C(TriageScore)[T.3.0],level 3.0 vs reference,4003,3633,370,-1.656,1.014,0.191,0.02615,1.394,0.1026
9,Unadjusted: Alcoholic,Alcoholic,Alcohol history,C(Alcoholic)[T.1.0],level 1.0 vs reference,4155,3785,370,0.7105,0.142,2.035,1.541,2.688,5.6e-07


## Adjusted logistic regression model

The adjusted model includes clinically relevant predictors selected before modeling: age, gender, key vital signs, neurologic status, triage score, and documented history indicators. Rows with missing values in any adjusted-model variable are omitted by complete case analysis.


In [6]:
adjusted_result, adjusted_results = fit_logistic_model(
    model_name="Adjusted clinical model",
    predictors=adjusted_predictors,
)

print(adjusted_result.summary())

adjusted_display = adjusted_results.copy()
for column in ["coefficient", "std_error", "odds_ratio", "ci_lower", "ci_upper", "p_value"]:
    adjusted_display[column] = adjusted_display[column].map(lambda value: f"{value:0.4g}")

display(adjusted_display)


                           Logit Regression Results                           
Dep. Variable:                Outcome   No. Observations:                 3352
Model:                          Logit   Df Residuals:                     3339
Method:                           MLE   Df Model:                           12
Date:                Tue, 09 Jun 2026   Pseudo R-squ.:                  0.5383
Time:                        14:15:22   Log-Likelihood:                -500.82
converged:                       True   LL-Null:                       -1084.8
Covariance Type:            nonrobust   LLR p-value:                1.317e-242
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -1.9855      2.825     -0.703      0.482      -7.522       3.551
C(Gender)[T.1]           -0.2219      0.242     -0.916      0.360      -0.697       0.253
C(Triage

,model,predictor,predictor_label,term,comparison,n_obs,events,non_events,coefficient,std_error,odds_ratio,ci_lower,ci_upper,p_value
1,Adjusted clinical model,Gender,Gender,C(Gender)[T.1],level 1 vs reference,3352,3019,333,-0.2219,0.2423,0.801,0.4982,1.288,0.3597
2,Adjusted clinical model,TriageScore,Triage score,C(TriageScore)[T.2.0],level 2.0 vs reference,3352,3019,333,-2.462,1.031,0.08525,0.0113,0.6431,0.01694
3,Adjusted clinical model,TriageScore,Triage score,C(TriageScore)[T.3.0],level 3.0 vs reference,3352,3019,333,-0.3428,1.051,0.7098,0.0905,5.567,0.7443
4,Adjusted clinical model,Alcoholic,Alcohol history,C(Alcoholic)[T.1.0],level 1.0 vs reference,3352,3019,333,-0.4029,0.2498,0.6684,0.4096,1.091,0.1068
5,Adjusted clinical model,Smoke,Smoking history,C(Smoke)[T.1.0],level 1.0 vs reference,3352,3019,333,0.08158,0.2312,1.085,0.6897,1.707,0.7242
6,Adjusted clinical model,FHCD,Family history of cardiac disease,C(FHCD)[T.1.0],level 1.0 vs reference,3352,3019,333,3.272,0.4132,26.38,11.73,59.28,2.386e-15
7,Adjusted clinical model,Age,Age,Age,per 1-year increase,3352,3019,333,-0.03002,0.007148,0.9704,0.9569,0.9841,2.681e-05
8,Adjusted clinical model,SBP,Systolic blood pressure,SBP,per 1 mmHg increase,3352,3019,333,0.04279,0.005101,1.044,1.033,1.054,4.988e-17
9,Adjusted clinical model,HR,Heart rate,HR,per 1 bpm increase,3352,3019,333,-0.09461,0.006375,0.9097,0.8984,0.9212,7.912e-50
10,Adjusted clinical model,RR,Respiratory rate,RR,per 1 breath/min increase,3352,3019,333,-0.09052,0.01389,0.9135,0.8889,0.9387,7.147e-11


## Save clean odds ratio results table

The final table combines the unadjusted and adjusted model terms. Values are rounded for reporting while retaining enough precision for review.


In [7]:
odds_ratio_results = pd.concat([unadjusted_results, adjusted_results], ignore_index=True)

rounded_numeric_columns = ["coefficient", "std_error", "odds_ratio", "ci_lower", "ci_upper"]
odds_ratio_results[rounded_numeric_columns] = odds_ratio_results[rounded_numeric_columns].round(6)

odds_ratio_results.to_csv(OUTPUT_PATH, index=False)
print(f"Saved odds-ratio results to: {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Results table shape: {odds_ratio_results.shape[0]:,} rows x {odds_ratio_results.shape[1]:,} columns")
display(odds_ratio_results.head(20))


Saved odds-ratio results to: reports\odds_ratio_results.csv
Results table shape: 24 rows x 14 columns


,model,predictor,predictor_label,term,comparison,n_obs,events,non_events,coefficient,std_error,odds_ratio,ci_lower,ci_upper,p_value
0,Unadjusted: Age,Age,Age,Age,per 1-year increase,5906,5053,853,-0.0518,0.0031,0.9495,0.9437,0.9554,0.0000
1,Unadjusted: SBP,SBP,Systolic blood pressure,SBP,per 1 mmHg increase,5906,5053,853,0.0324,0.0020,1.0329,1.0289,1.0369,0.0000
2,Unadjusted: HR,HR,Heart rate,HR,per 1 bpm increase,5906,5053,853,-0.0761,0.0028,0.9267,0.9217,0.9317,0.0000
3,Unadjusted: RR,RR,Respiratory rate,RR,per 1 breath/min increase,5906,5053,853,-0.1340,0.0071,0.8746,0.8625,0.8867,0.0000
4,Unadjusted: SpO2,SpO2,Oxygen saturation,SpO2,per 1 percentage-point increase,5906,5053,853,0.4088,0.0196,1.5050,1.4481,1.5641,0.0000
5,Unadjusted: GCS,GCS,Glasgow Coma Scale,GCS,per 1-point increase,5906,5053,853,-0.0247,0.0036,0.9756,0.9689,0.9824,0.0000
6,Unadjusted: Gender,Gender,Gender,C(Gender)[T.1],level 1 vs reference,5906,5053,853,-0.0133,0.0815,0.9868,0.8411,1.1577,0.8703
7,Unadjusted: TriageScore,TriageScore,Triage score,C(TriageScore)[T.2.0],level 2.0 vs reference,4003,3633,370,-3.5847,1.0043,0.0277,0.0039,0.1986,0.0004
8,Unadjusted: TriageScore,TriageScore,Triage score,C(TriageScore)[T.3.0],level 3.0 vs reference,4003,3633,370,-1.6557,1.0143,0.1910,0.0262,1.3943,0.1026
9,Unadjusted: Alcoholic,Alcoholic,Alcohol history,C(Alcoholic)[T.1.0],level 1.0 vs reference,4155,3785,370,0.7105,0.1420,2.0350,1.5407,2.6878,0.0000


## Interpretation cautions

- Odds ratios above 1 indicate higher odds of `Outcome = 1` for a one-unit increase in a continuous predictor or for the displayed categorical level compared with its reference level.
- Odds ratios below 1 indicate lower odds of `Outcome = 1` under the same comparison.
- These are associational estimates, not causal effects. The observed associations may reflect confounding, selection bias, measurement timing, repeated patient observations, missing-data mechanisms, or other design factors.
- The direction and clinical meaning of coded variables such as `Outcome`, `Gender`, and `TriageScore` should be confirmed from source documentation before clinical reporting.


## Conclusion 

This notebook examined which clinical variables were associated with the cardiac outcome using logistic regression and odds ratios. The adjusted model was the most useful because it compared each variable while controlling for the other predictors in the model. After adjustment, family history of cardiac disease had the strongest positive association with the outcome. Higher systolic blood pressure and higher oxygen saturation were also linked with greater odds of `Outcome = 1`. Older age, higher heart rate, higher respiratory rate, and triage score level 2 were linked with lower odds of the outcome.

Some variables, including gender, smoking history, alcohol history, triage score level 3, and GCS, were not statistically significant after adjustment. This suggests that their individual effects may be weaker once the other clinical variables are included. The odds ratio analysis helped identify which factors had the clearest relationship with the outcome. These results should still be interpreted carefully because the model shows association, not causation, and the meaning of the outcome variable should be confirmed before making clinical conclusions.